# Getting Started with minicamels

This notebook introduces the `minicamels` dataset — a curated subset of the
[CAMELS-US](https://ral.ucar.edu/solutions/products/camels) large-sample
hydrometeorological dataset designed for teaching.

**What's in the dataset:**
- 50 USGS basins spanning diverse climate regions across the contiguous US
- 30 water years of daily data (WY1981–WY2010)
- Daymet-derived meteorological forcings: precipitation, temperature, solar radiation, vapor pressure
- USGS observed streamflow, normalized to mm/day
- Static catchment attributes: area, elevation, aridity, soil properties, land cover, and more

---

## Setup

**Google Colab users:** uncomment and run the cell below first.

In [ ]:
# Colab only — skip if running locally
# !pip install git+https://github.com/andrbenn/minicamels.git

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minicamels import MiniCamels

ds = MiniCamels()
print(ds)

---
## 1. Exploring the basin index and attributes

In [ ]:
# All 50 basins
basins = ds.basins()
print(f"{len(basins)} basins")
basins

In [ ]:
# Static catchment attributes — one row per basin
attrs = ds.attributes()
attrs.head()

In [ ]:
# Summary statistics across the 50 basins
attrs[["area_km2", "mean_prcp", "aridity", "q_mean", "runoff_ratio"]].describe().round(2)

In [ ]:
# Map of basin locations, coloured by aridity
# aridity = PET / P:  < 1 = energy-limited (humid),  > 1 = water-limited (arid)
ds.plot_map(color_by="aridity")
plt.show()

---
## 2. Loading a single basin

Each basin's timeseries is an `xarray.Dataset` with a `time` dimension and six variables.

In [ ]:
basin_id = "01013500"  # Fish River near Fort Kent, Maine

ts = ds.load_basin(basin_id)
ts

In [ ]:
# Variables, units, and coverage
for name, var in ts.data_vars.items():
    n_nan = int(np.isnan(var.values).sum())
    pct = 100 * n_nan / len(var)
    print(f"  {name:6s}  units={var.attrs.get('units',''):10s}  missing={pct:.1f}%")

In [ ]:
# Full timeseries plot — one panel per variable
ds.plot_basin(basin_id)
plt.show()

---
## 3. Slicing by time and water year

The hydrological water year runs from **October 1 to September 30**.
Water year 2000 = Oct 1 1999 – Sep 30 2000.

In [ ]:
# One water year
wy2000 = ds.get_water_year(basin_id, water_year=2000)
print(f"WY2000: {wy2000.time.values[0]} → {wy2000.time.values[-1]}  ({len(wy2000.time)} days)")

In [ ]:
# Plot precipitation and streamflow for a single water year
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

wy2000["prcp"].plot(ax=ax1, color="steelblue", linewidth=0.8)
ax1.set_ylabel("Precipitation (mm/day)")
ax1.set_title(f"{basin_id} — Water Year 2000")

wy2000["qobs"].plot(ax=ax2, color="seagreen", linewidth=0.8)
ax2.set_ylabel("Streamflow (mm/day)")
ax2.set_xlabel("")

fig.tight_layout()
plt.show()

In [ ]:
# Or slice by date string
summer = ds.get_forcings(basin_id, start="1995-06-01", end="1995-08-31")
print(f"Summer slice: {len(summer.time)} days")

---
## 4. Comparing multiple basins

Load several basins at once and concatenate them along a `basin` dimension.

In [ ]:
# Pick a humid northeastern basin and an arid southwestern one
humid  = "01013500"  # Fish River, Maine       (aridity ~ 0.63)
arid   = "08194200"  # Frio River, Texas       (aridity ~ 2.1)

multi = ds.open_basins([humid, arid])
multi

In [ ]:
# Mean annual streamflow for each basin
q_annual = multi["qobs"].resample(time="YE").sum()

fig, ax = plt.subplots(figsize=(10, 4))
for bid in [humid, arid]:
    name = basins.set_index("basin_id").loc[bid, "basin_name"]
    q_annual.sel(basin=bid).plot(ax=ax, label=f"{bid}\n{name}")

ax.set_title("Annual streamflow: humid vs. arid basin")
ax.set_ylabel("Streamflow (mm/year)")
ax.set_xlabel("")
ax.legend(fontsize=8)
plt.show()

In [ ]:
# Mean seasonal cycle (all 50 basins)
all_basins = ds.open_basins()
seasonal_q = all_basins["qobs"].groupby("time.month").mean()

fig, ax = plt.subplots(figsize=(8, 4))
for bid in seasonal_q.basin.values:
    ax.plot(seasonal_q.month, seasonal_q.sel(basin=bid),
            color="steelblue", alpha=0.25, linewidth=0.8)

seasonal_q.mean("basin").plot(ax=ax, color="navy", linewidth=2, label="50-basin mean")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
ax.set_ylabel("Mean daily streamflow (mm/day)")
ax.set_title("Seasonal streamflow cycle — all 50 basins")
ax.legend()
plt.show()

---
## 5. Preparing data for machine learning

A common task is predicting streamflow from meteorological inputs.
Here we build a simple feature matrix `X` and target vector `y` for one basin.

In [ ]:
import pandas as pd

basin_id = "01013500"
ts = ds.load_basin(basin_id)

# Convert to a pandas DataFrame for easy manipulation
df = ts.to_dataframe().drop(columns=["spatial_ref"], errors="ignore")
df.head()

In [ ]:
# Drop rows where streamflow is missing, then split features and target
df = df.dropna(subset=["qobs"])

forcing_cols = ["prcp", "tmax", "tmin", "srad", "vp"]
X = df[forcing_cols].values   # shape: (n_days, 5)
y = df["qobs"].values         # shape: (n_days,)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")

In [ ]:
# Conventional train/test split at a water-year boundary
# Train: WY1981–WY2000  |  Test: WY2001–WY2010
split_date = "2000-10-01"
train_mask = df.index < split_date
test_mask  = df.index >= split_date

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"Train: {train_mask.sum()} days  |  Test: {test_mask.sum()} days")

In [ ]:
# Quick baseline: predict with mean training streamflow
y_pred_baseline = np.full_like(y_test, y_train.mean())

# Nash-Sutcliffe Efficiency (NSE) — standard metric in hydrology
# NSE = 1.0 is perfect;  NSE = 0 means model is no better than the mean
def nse(obs, sim):
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - obs.mean()) ** 2)

print(f"Baseline NSE (mean prediction): {nse(y_test, y_pred_baseline):.3f}")
print()
print("Next step: fit a model (e.g. random forest, LSTM) and compare!")

---
## Next steps

Some things to try:

- **Add lag features** — streamflow today depends on rainfall from the past several days.
  Try adding `prcp` lagged by 1–14 days as additional columns in `X`.
- **Use catchment attributes as static inputs** — combine the time-varying forcings
  with static attributes (e.g. `area_km2`, `aridity`, `frac_forest`) to build a model
  that generalises across basins.
- **Try a sequence model** — daily hydrology has strong temporal memory.
  A rolling-window approach or an LSTM reads the data as sequences rather than
  independent daily snapshots.
- **Evaluate across all 50 basins** — train on some basins, test on held-out ones
  (regionalisation / transfer learning).